In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import re
from matplotlib.lines import Line2D



In [2]:
import sys
sys.path.append('../../../Code')
from Run_ml_model import make_sorted_target_folds,make_one_per_group_folds, run_models


### Run using one subset of the data
The results of all the datasets are runing on high-performance computing (HPC) clusters (ISAAC in University of Tennessee) to improve computational efficiency, and are saved in **../results**.


In [3]:
df0 = pd.read_csv('../data/metrics_masks5_plus_full.csv')
df0 = df0.drop(columns=["image_path", "mask_path","scale_max_used"])
feature_cols = df0.columns[4:]

df0["Fungal_Strain"] = df0["plate"].str.split("_").str[1]
df0["Nitrogen_Level"] = df0["plate"].str.split("_").str[2]

views = df0['view'].unique()
mask_types = df0['mask_type'].unique()
dates = df0['date'].unique()[1:]


In [4]:
plates = df0["plate"].unique()
df_plate = (
    df0[["plate", "Fungal_Strain", "Nitrogen_Level"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
outplate_class = make_one_per_group_folds(df_plate, 
                                          group_cols=list(("Fungal_Strain", "Nitrogen_Level")),
                                          n_splits=5, random_state=42)

In [5]:
test_plates_folds = []
for _, test_plate_idx in outplate_class:
    test_plates  = df_plate.iloc[test_plate_idx]["plate"].values
    test_plates_folds.append(test_plates)
print("Created test_plates_folds for CV based on plates.")
print(test_plates_folds)


Created test_plates_folds for CV based on plates.
[array(['18_FG_N-10_4_MV.003', '25_FG_N-100_2_MV.003',
       '27_LE_N-10_5_MV.003', '29_LE_N-100_4_MV.003', '3_FG_N-1_2_MV.003',
       '9_LE_N-1_5_MV.003'], dtype=object), array(['11_FG_N-1_4_MV.003', '19_LE_N-1_2_MV.003', '1_FG_N-10_3_MV.003',
       '21_FG_N-100_5_MV.003', '30_LE_N-100_3_MV.003',
       '7_LE_N-10_2_MV.003'], dtype=object), array(['12_FG_N-1_5_MV.003', '14_LE_N-100_5_MV.003',
       '15_LE_N-10_1_MV.003', '23_FG_N-100_4_MV.003', '6_LE_N-1_4_MV.003',
       '8_FG_N-10_2_MV.003'], dtype=object), array(['13_FG_N-1_1_MV.003', '17_LE_N-1_3_MV.003', '20_FG_N-10_5_MV.003',
       '24_FG_N-100_3_MV.003', '26_LE_N-100_2_MV.003',
       '28_LE_N-10_4_MV.003'], dtype=object), array(['10_LE_N-1_1_MV.003', '16_LE_N-10_3_MV.003',
       '22_FG_N-100_1_MV.003', '2_LE_N-100_1_MV.003',
       '4_FG_N-10_1_MV.003', '5_FG_N-1_3_MV.003'], dtype=object)]


In [6]:
def run_single_combination(view, mask_type, date, df_combined, feature_cols, test_plates_folds):
    subset = df_combined[(df_combined['view'] == view) & 
                         (df_combined['mask_type'] == mask_type) & 
                         (df_combined['date'] == date)]
    if subset.empty:
        return None

    print(f"=========Starting: {view}, {mask_type}, {date}======")
    results = run_models(
        subset, feature_cols,
        target_col="Nitrogen_Level",
        group_cols=("Fungal_Strain", "Nitrogen_Level"),
        outer_splits=5, 
        outer_plate_folds=test_plates_folds,
        inner_splits=4,
        random_state=42,n_jobs=1)  # Note: Set to 1 here as parallelization is handled at the outer loop level.
    return (view, mask_type, date), results

In [7]:
# The one (view, mask_type, date) combination this notebook inspects.
# run_single_combination returns its results keyed by exactly this tuple, and the HPC
# results pickle is keyed the same way
# so the local run and the server-output lookup have to agree 
subset_key = ('overhead', 'all', '2024-12-21')
VIEW, MASK_TYPE, DATE = subset_key


In [8]:
results_all_raw_1221 = run_single_combination(*subset_key, df0, feature_cols=feature_cols, test_plates_folds=test_plates_folds)


=========Starting: overhead, all, 2024-12-21======


/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in ver


=== ElasticNet_MultinomialLR ===
Pooled confusion matrix (rows=true, cols=pred):
       N-1  N-10  N-100
N-1      7     1      2
N-10     1     6      3
N-100    3     3      4


/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in ver


=== Lasso_MultinomialLR ===
Pooled confusion matrix (rows=true, cols=pred):
       N-1  N-10  N-100
N-1      8     1      1
N-10     1     6      3
N-100    3     2      5

=== ShrinkageLDA ===
Pooled confusion matrix (rows=true, cols=pred):
       N-1  N-10  N-100
N-1      6     1      3
N-10     0     6      4
N-100    2     2      6


/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/menglinghe/Library/Mobile Documents/com~apple~CloudDocs/UTK/GRA-UTK/fungus-nitrogen/fungus_nitrogen_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in ver


=== PLSDA_PLSScores_plus_LR ===
Pooled confusion matrix (rows=true, cols=pred):
       N-1  N-10  N-100
N-1      7     1      2
N-10     2     5      3
N-100    2     2      6

=== RandomForest ===
Pooled confusion matrix (rows=true, cols=pred):
       N-1  N-10  N-100
N-1      8     0      2
N-10     2     7      1
N-100    1     2      7

=== Comparison ===
                          mean_accuracy  mean_balanced_accuracy
RandomForest                   0.733333                0.733333
Lasso_MultinomialLR            0.633333                0.633333
ShrinkageLDA                   0.600000                0.600000
PLSDA_PLSScores_plus_LR        0.600000                0.600000
ElasticNet_MultinomialLR       0.566667                0.566667


### Check with server output

In [9]:
with open("../results/nitrogen_class/results_nitrogen_update.pkl", "rb") as f:
    results_nitrogen_class = pickle.load(f)

print(type(results_nitrogen_class))   # should be dict

<class 'dict'>


In [10]:
example = results_nitrogen_class.get(subset_key)
if example is None:
    raise KeyError(f"{subset_key} not in the results pickle; available keys e.g. "
                   f"{list(results_nitrogen_class)[:3]}")

In [11]:
print(example[2])

                          mean_accuracy  mean_balanced_accuracy
RandomForest                   0.733333                0.733333
Lasso_MultinomialLR            0.633333                0.633333
ShrinkageLDA                   0.600000                0.600000
PLSDA_PLSScores_plus_LR        0.600000                0.600000
ElasticNet_MultinomialLR       0.566667                0.566667
